<a href="https://colab.research.google.com/github/KruthikaVirupakshappa/DATA266-Lab1-Pair37/blob/jiayi-work/task1_llm/jiayi/src/Task1_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
SID4         =  4253 # last four digits of your SJSU ID
SEED         = SID4        # torch.manual_seed, np.random.seed, random_state
SLICE        = SID4 % 1000   # start index of your dataset subset
HP_ID        = SID4 % 6      # your assigned hyperparameter arm, Table 0.1
CLS_A, CLS_B = (SID4 % 10), ((SID4 // 10) % 10)   # your two focus classes

In [2]:
print(f"SID4: {SID4}, SEED: {SEED}, SLICE: {SLICE}, HP_ID: {HP_ID}, CLS_A, CLS_B: {CLS_A}, {CLS_B}")

SID4: 4253, SEED: 4253, SLICE: 253, HP_ID: 5, CLS_A, CLS_B: 3, 5


## Task 1
### data preparation

In [26]:
import random
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [21]:
# load shared data from download_tinystories.py
from datasets import load_from_disk
from pathlib import Path
from datasets import load_dataset

dataset_folder = Path("task1_llm/data/tinystories")
data = load_from_disk(str(dataset_folder))

# inspect the original dataset
print(data)
print(data.keys())

# only use the original training set for training and validation
original_train = data["train"]

# use SEED to randomize original_train
shuffled_train = original_train.shuffle(seed=SEED)

# split the shuffled train
my_train = shuffled_train.select(range(100_000))
my_val = shuffled_train.select(range(100_000, 110000))

# inspect the splitted dataset by checking shape and a few data
print(len(my_train))
print(len(my_val))
print(my_train.column_names)
print(my_train[0])
print(my_val[0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
dict_keys(['train', 'validation'])
100000
10000
['text']
{'text': 'Once upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a little girl named Lily. \n\n"Hello, Blue! You can fly very fast!" said Lily. \n\n"Yes, I can fly very fast!" replied Blue, happily. \n\nBut then, a big storm came and Blue got lost. Blue was scared and didn\'t know what to do. \n\n"Can you help me, Lily?" asked Blue. \n\n"Yes, I can help you!" said Lily. "Let\'s look for your history book. Maybe it will help us find your way home." \n\nSo, Lily and Blue searched for Blue\'s history book. They found it and read about all the places Blue had flown before. They followed the book and finally, Blue found his way home. \n\n"Thank you, Lily! You\'re my hero!" said Blue. \

In [24]:
train_text = "\n".join(my_train["text"])
val_text = "\n".join(my_val["text"])

chars = sorted(list(set(train_text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

def encode(s):
    return [char_to_idx[c] for c in s]

def decode(ids):
    return "".join(idx_to_char[i] for i in ids)

unseen_chars = set(val_text) - set(chars)
print("Unseen validation chars:", unseen_chars)

Unseen validation chars: set()


In [25]:
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print("Vocab size:", vocab_size)
print("Train characters:", len(train_data))
print("Val characters:", len(val_data))

Vocab size: 152
Train characters: 89734384
Val characters: 8964594


In [27]:
class CharDataset(Dataset):
    def __init__(self, data, seq_len=128):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len

        x = self.data[start : start + self.seq_len]
        y = self.data[start + 1 : start + self.seq_len + 1]

        return x, y

In [28]:
seq_len = 128
train_dataset = CharDataset(train_data, seq_len)
val_dataset = CharDataset(val_data, seq_len)

print(len(train_dataset))
print(len(val_dataset))

701049
70035


In [32]:
x, y = train_dataset[0]

print(x.shape)
print(y.shape)
print("Input:\n" + decode(x.tolist()))
print("Output:\n" + decode(y.tolist()))

torch.Size([128])
torch.Size([128])
Input:
Once upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a l
Output:
nce upon a time, there was a little bird named Blue. Blue was very fast and loved to fly high in the sky. One day, Blue met a li


In [ ]:
generator = torch.Generator()
generator.manual_seed(SEED)
loader = DataLoader(dataset, batch_size=32, shuffle=True, generator=generator)

In [ ]:
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=32,shuffle=True, generator=generator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)